# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Runner

**Phase 1 V3: Magnitude L1 Sparsity + Rebalanced Shots + Converged Training**

Pipeline: `Calibration → S1 (V3 Typed Training) → S5 (Contract v3 Deployment Calibration) → Benchmark (Baseline vs Fixed-Control)`

**Target:** SteerNet [13, 20, 22] achieves ≥ +15.0 pp zero-shot gain on GSM8K with zero heuristic dials, and non-negative delta on 2-shot & 4-shot (eliminate regression).

**V3 Changes vs V2:**
- `STOP_RE` fix: correctly stops on `"Final answer: X units.\n"` patterns
- Magnitude L1 sparsity penalty (`mag_l1_weight=0.02`): forces selective steering, fixes 4-shot regression
- Shot distribution rebalanced: `[0.30, 0.10, 0.25, 0.10, 0.25]` — more few-shot coverage
- Lambda redistribution: `result=0.40, final=0.35` (75% weight on critical tokens)
- 5 epochs (was 4) — crit still declining at epoch 3
- Full dataset: `n_samples=7473` (was 7000)


## 1. Environment & Hardware Check

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy tqdm

## 2. Clone Repository (`development` branch)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")
# Pre-calibrated norms_calibration.json is tracked in the repository.
# If missing, calibrate 500 samples automatically.
if not os.path.exists("norms_calibration.json"):
    !python scripts/calibrate_norms.py --n_samples 500 --output norms_calibration.json
else:
    print("✅ Certified norms_calibration.json found in repository:")
!ls -lh norms_calibration.json


## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache


In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration (Phase 1 V2 — Zero Heuristic Dials)


In [ ]:
# ==============================================================================
# ARS EXPERIMENT SCALE & HYPERPARAMETER CONFIGURATION
# Phase 1 V3: Magnitude L1 Sparsity + Rebalanced Shots + Converged Training
# ==============================================================================
# Select execution profile:
#   'balanced' (Recommended): 3000 train rows, 300 test items (~22-25 min)
#   'medium'                : 3000 train rows, 100 test items (~15 min)
#   'full'                  : 7473 train rows, 500 test items (~40 min)
#   'quick'                 : smoke test (~2 min)
PROFILE = 'balanced'
S1_OBJECTIVE = 'nll'
LOSS_MODE = 'typed'       # V2 typed loss weights result+final tokens at 75%

# ========== PHASE 1 V3 STEERING PHYSICS ==========
STEER_KAPPA  = 0.04       # Certified amplitude (C_l = kappa * N_l)
K_SCALE_MODE = 'sqrt'     # Multi-layer budget: active_k^(-0.5)
STEER_SCALE  = 0.30       # Uniform scale across ALL shot counts (no heuristic dials)

# V3 Optimization settings
BATCH_SIZE = 1
GRAD_ACCUM = 16            # Effective batch size = 16
MAX_TOK_LEN = 1024        # 1024-token context for up to 4-shot mixture

if PROFILE == 'quick':
    S1_EPOCHS    = 2;  S1_LR = 2e-4;  S1_TRAIN_N = 50
    S1_VAL_N     = 10; CALIB_SEL_N = 5
    BENCH_N_TEST = 10; BENCH_SHOTS = '0'
elif PROFILE == 'balanced':
    S1_EPOCHS    = 5;  S1_LR = 2e-4;  S1_TRAIN_N = 3000
    S1_VAL_N     = 100; CALIB_SEL_N = 60
    BENCH_N_TEST = 300; BENCH_SHOTS = '0 2 4'
elif PROFILE == 'medium':
    S1_EPOCHS    = 5;  S1_LR = 2e-4;  S1_TRAIN_N = 3000
    S1_VAL_N     = 100; CALIB_SEL_N = 60
    BENCH_N_TEST = 100; BENCH_SHOTS = '0 2 4'
elif PROFILE == 'full':
    S1_EPOCHS    = 5;  S1_LR = 2e-4;  S1_TRAIN_N = 7473
    S1_VAL_N     = 120; CALIB_SEL_N = 120
    BENCH_N_TEST = 500; BENCH_SHOTS = '0 2 4'

print('='*80)
print('✅ PHASE 1 V3 CONFIGURATION LOADED')
print('='*80)
print(f'Profile: {PROFILE} | Objective: {S1_OBJECTIVE} | Loss Mode: {LOSS_MODE}')
print(f'\n🔧 V3 Architecture & Training Changes:')
print(f'  • Magnitude L1 Sparsity Penalty (mag_l1_weight=0.02): selective steering')
print(f'  • Shot Distribution: [0.30, 0.10, 0.25, 0.10, 0.25] — more few-shot coverage')
print(f'  • Lambda Redistribution: result=0.40, final=0.35 (75% on critical tokens)')
print(f'  • Epochs: 5 (was 4) — training crit still declining at epoch 3')
print(f'  • Full dataset: n_samples=7473 (was 7000)')
print(f'  • STOP_RE fix: handles "Final answer: X units.\\n" patterns')
print(f'\n📊 Training Config:')
print(f'  • S1: {S1_EPOCHS} epochs, {S1_TRAIN_N} samples, peak_lr={S1_LR}')
print(f'  • Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM} (effective {BATCH_SIZE * GRAD_ACCUM})')
print(f'  • Calibration: {CALIB_SEL_N} samples')
print(f'  • Benchmark: {BENCH_N_TEST} test items, shots=[{BENCH_SHOTS}]')
print(f'\n🎯 Target: >=+15.0 pp 0-shot gain | 2-shot & 4-shot delta >= 0 pp')
print('='*80)


## 7. Fixed Layer Configuration (Phase 1 V2)


In [ ]:
# ==============================================================================
# FIXED LAYER CONFIGURATION (Phase 1 V2)
# ==============================================================================
FIXED_LAYERS = [13, 15, 18, 20, 22]
S_FIXED = '13,20,22'  # Proven 3-layer synergy: L13 (early), L20 (mid-late), L22 (late anchor)

# --- Steering Physics V2 ---
# [1] Prefill Steering: ACTIVE (0-shot KV priming; few-shot demos shielded)
# [2] Few-Shot Prompt Shielding: ACTIVE via shield_policy='context'
# [3] Uniform Scaling: NO shot-aware knobs (purged in V2)
# [4] Contract v3: No shot_scales field; apply_contract sets steer_scale only

print('='*80)
print('✅ FIXED LAYER CONFIGURATION (V2)')
print('='*80)
print(f'Fixed candidate layers: {FIXED_LAYERS}')
print(f'Active fixed route: {[13, 20, 22]}')
print(f'Enforced 3-layer subset (S_FIXED): {S_FIXED}')
print(f'\n🔬 Steering Physics (Phase 1 V2):')
print(f'  • ✅ Question Prefill Steering: ACTIVE')
print(f'  • ✅ Few-Shot Demo Shielding: ACTIVE (shield_policy=context)')
print(f'  • ✅ Uniform Scale={STEER_SCALE}: No shot-dependent heuristics')
print(f'  • ✅ S5 Envelope: Contract v3, kappa={STEER_KAPPA}')
print('='*80)


## 8. Stage S1: SteerNet Training (V2 Typed Loss)


In [ ]:
# Stage S1: SteerNet Training with V2 Typed Reasoning Loss & Multi-Context Mixture
import subprocess
command = f'''python scripts/run_ars_pipeline.py --stage S1 {STEERNET_ARCH_ARGS} \
    --objective {S1_OBJECTIVE} \
    --s1_epochs {S1_EPOCHS} \
    --s1_lr {S1_LR} \
    --s1_train_n {S1_TRAIN_N} \
    --s1_val_n {S1_VAL_N} \
    --batch_size {BATCH_SIZE} \
    --steer_scale {STEER_SCALE} \
    --s_fixed {S_FIXED} \
    --set steer_kappa={STEER_KAPPA} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set grad_accum={GRAD_ACCUM} \
    --set max_tok_len={MAX_TOK_LEN} \
    --set loss_mode={LOSS_MODE}'''
process = subprocess.Popen(
    command, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(
        f"S1 pipeline failed with exit code {process.returncode}; "
        "full runner output is shown above."
    )

print('\n' + '='*80)
print('✅ S1 Training Complete')
print('='*80)


## 9. Stage S5: Deployment Calibration (Contract v3)


In [ ]:
# Stage S5: Calibration — writes deployment_contract.json (version 3, no shot_scales)
!python scripts/run_ars_pipeline.py --stage S5 {STEERNET_ARCH_ARGS} \
    --calib_sel_n {CALIB_SEL_N} \
    --contract deployment_contract.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set k_scale_mode={K_SCALE_MODE}

import json as _json, pathlib as _pl
_c = _json.loads(_pl.Path("deployment_contract.json").read_text())
print(f"\nContract version: {_c.get('version')}  (expected 3)")
assert _c.get("version") == 3, "Expected contract v3 — shot_scales were not purged!"
assert "shot_scales" not in _c, "shot_scales field must not appear in v3 contract!"
print("\n" + "="*80)
print("✅ S5 Calibration Complete (Contract v3 verified)")
print("="*80)


## 10. Benchmark: Baseline vs Fixed-Control (Zero Heuristic Dials)


In [ ]:
# Benchmark: Baseline vs Fixed-control [13,20,22]
# Zero heuristic dials: model executes with identical weights and scale across ALL shots
!python scripts/run_ars_pipeline.py --stage bench {STEERNET_ARCH_ARGS} \
    --contract deployment_contract.json \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --record experiment_record.json \
    --report \
    --arms baseline,fixed \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE}

print('\n' + '='*80)
print('✅ Benchmark Complete')
print('='*80)


## 11. Inspect Benchmark Report


In [ ]:
import os, json
from IPython.display import display
from src.evaluation.diagnostics import plot_paired, plot_training, show_generations

# Report is now written to report/REPORT_GSM8K.md (suite-scoped) and report/REPORT.md (alias)
for report_path in ["report/REPORT_GSM8K.md", "report/REPORT.md"]:
    if os.path.exists(report_path):
        print(f"--- {report_path} ---")
        with open(report_path, "r", encoding="utf-8") as f:
            print(f.read())
        break
else:
    print("Report not found. Check benchmark logs above.")

if os.path.exists("report/training_log.json"):
    with open("report/training_log.json", encoding="utf-8") as f:
        hist = json.load(f)
    plot_training(hist, "report/training_curves.png")
    # Print V2 checkpoint metrics
    print(f"\nBest crit: {hist.get('best_crit', 'N/A')}")
    for ep_info in hist.get("epochs", []):
        if isinstance(ep_info, dict):
            print(f"  crit={ep_info.get('crit', '?'):.4f}  result_top1={ep_info.get('result_top1', '?'):.3f}  val_nll={ep_info.get('validation_nll', '?'):.4f}")

if os.path.exists("experiment_record.json"):
    with open("experiment_record.json", encoding="utf-8") as f:
        record = json.load(f)
    plot_paired(record, "report/paired_outcomes.png")
    # Print acceptance-criteria check
    print("\n\u2705 Acceptance Criteria Check:")
    shots_data = record.get("shots", {})
    for shot_str, shot_data in shots_data.items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        b_acc = b.get("lenient_accuracy", 0.0)
        s_acc = s.get("lenient_accuracy", 0.0)
        delta = s_acc - b_acc
        paired = shot_data.get("paired", {})
        print(f"  {shot_str}-shot | Baseline={b_acc:.1f}% | Fixed={s_acc:.1f}% | Delta={delta:+.1f} pp | Wins={paired.get('fixed_wins',0)} Losses={paired.get('fixed_losses',0)}")
        if paired.get("taxonomy"):
            print(f"    Taxonomy: {paired['taxonomy']}")
    print(json.dumps(record, indent=2))

# Per-arm generation files: gsm8k_{n}shot_{arm}.jsonl
for shot_str in shots_data.keys():
    gen_path = f"report/generations/gsm8k_{shot_str}shot_fixed.jsonl"
    if os.path.exists(gen_path):
        display(show_generations(gen_path, k=4, flips_only=False))


## 12. Save Artifacts


In [ ]:
import shutil, os, json

os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)
for fname in [
    "deployment_contract.json",
    "experiment_record.json",
    "norms_calibration.json",
    "checkpoints/steernet.pt",
]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
        print(f"Copied: {fname}")
if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
    print("Copied: report/")

# Verify contract version in artifact
_c_path = "/kaggle/working/ars_artifacts/deployment_contract.json"
if os.path.exists(_c_path):
    _c = json.loads(open(_c_path).read())
    print(f"\nArtifact contract version: {_c.get('version')}")
    assert _c.get("version") == 3, "Contract must be v3 in artifact!"
    assert "shot_scales" not in _c, "shot_scales must not be in artifact contract!"
    print("\u2705 Contract v3 artifact verified")

print("\nArtifacts saved to /kaggle/working/ars_artifacts:")
!ls -lh /kaggle/working/ars_artifacts
